In [ ]:
import torch
import math
import tqdm
import wandb
import datasets
from torch.utils.data import Dataset, DataLoader
from collections import Counter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class Attention(torch.nn.Module):

  def __init__(self, d_model, num_heads ):
    self.num_heads = num_heads

    self.d_model = d_model
    super().__init__()
    self.W_q = torch.nn.Linear(d_model, d_model)
    self.W_k = torch.nn.Linear(d_model, d_model)
    self.W_v = torch.nn.Linear(d_model, d_model)
    self.W_o = torch.nn.Linear(d_model, d_model)

  def softmax(self,x, dim = -1):
    x_max = torch.max(x, dim = dim , keepdim = True)[0]
    rescaled = x - x_max
    exp = torch.exp(rescaled)
    sum = torch.sum(exp, dim = dim, keepdim = True)
    return exp/sum

  def scaled_dot_product_attention(self,Q,K,V,mask = None):
    """
    For scaled_dot_product_attention

    args:
    Q : query vector : shape(batch_size, num_heads, seq_len, d_q)
    K : key vector : shape(batch_size, num_heads, seq_len,d_k)
    V : value vector : shape(batch_size, num_heads, seq_len, d_v)

    returns:
    attention_product : shape(batch_size, num_heads, seq_len, d_v)

    usually d_k = d_v = d_q

    """
    d_k = K.shape[-1]

    dot_product = Q@K.transpose(-2,-1)
    scaled_dot_product = dot_product/math.sqrt(d_k)
    if self.mask is not None:
      scaled_dot_product = scaled_dot_product.masked_fill(mask == 0, -float("inf"))
    attention_weights = self.softmax(scaled_dot_product)
    attention_product = attention_weights@V
    return attention_product, attention_weights

  def forward(self, Q, K, V,mask = None,use_cache=False, kv_cache = None):
    """
    Computes the multi-head attention for the given query, key, and value vectors. (scaled_attention across num_heads projections)

    args:
    d_model : dimension of the model
    num_heads : number of heads to use in the multi-head attention

    Q : query vector -> shape(batch_size,seq_len, d_model)
    K : key vector -> shape(batch_size, seq_len, d_model)
    V : value vector -> shape(batch_size, seq_len, d_model)

    model -> shape (d_model x d_model)
    output_Q -> shape(batch_size, seq_len, d_model)
    output_K -> shape(batch_size, seq_len, d_model)
    output_V -> shape(batch_size, seq_len, d_model

    project Q_i -> shape(batch_size, num_heads, seq_len, d_q)
    project K_i -> shape(batch_size, num_heads, seq_len, d_k)
    project V_i -> shape(batch_size, num_heads, seq_len, d_v)

    re-shape Q -> shape(batch_size, num_heads, seq_len, d_q)
    re-shape K -> shape(batch_size, num_heads, seq_len, d_k)
    re-shape V -> shape(batch_size, num_heads, seq_len, d_v)

    output model -> shape(d_model,d_model)

    returns:
    output -> shape(batch_size, seq_len, d_model)
    """
    batch_size, seq_len, _ = Q.shape

    self.d_k = self.d_model//self.num_heads
    self.d_v = self.d_model//self.num_heads
    self.d_q = self.d_model//self.num_heads



    output_Q = self.W_q(Q)
    output_K = self.W_k(K)
    output_V = self.W_v(V)

    Q_proj = output_Q.view(batch_size, -1, self.num_heads, self.d_q)
    K_proj = output_K.view(batch_size, -1, self.num_heads, self.d_k)
    V_proj = output_V.view(batch_size, -1, self.num_heads,self.d_v)

    Q_proj = Q_proj.transpose(1,2)
    K_proj = K_proj.transpose(1,2)
    V_proj = V_proj.transpose(1,2)

    if use_cache:
      if kv_cache is not None:
        past_K, past_V = kv_cache
        K_proj = torch.cat([past_K, K_proj], dim = -2)
        V_proj = torch.cat([past_V, V_proj], dim = -2)


      new_kv_cache = (K_proj, V_proj)
    else:
      new_kv_cache = None

    is_causal = True if (kv_cache is None and seq_len > 1) else False

    self.mask = mask
    O_proj = torch.nn.functional.scaled_dot_product_attention(
    Q_proj, K_proj, V_proj,
    attn_mask=None,
    is_causal=is_causal
)
    O_proj = O_proj.transpose(1,2)
    O = O_proj.reshape(batch_size, -1, self.d_model)

    output = self.W_o(O)

    return output,new_kv_cache


In [ ]:
class LayerNorm(torch.nn.Module):

  def __init__(self, d_model,eps = 1e-5):
    super().__init__()
    self.eps = eps
    self.gamma = torch.nn.Parameter(torch.ones(d_model))
    self.beta = torch.nn.Parameter(torch.zeros(d_model))

  def forward(self,x,dim = -1):

    mean = torch.mean(x, dim = dim, keepdim = True)
    var = torch.var(x, dim = dim, keepdim = True, unbiased = False)

    numerator = x - mean
    denominator = torch.sqrt(var + self.eps)

    return numerator/denominator * self.gamma + self.beta

In [ ]:
class PositionalEncodings(torch.nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()
        pe = torch.zeros(max_seq_len, d_model)
        positions = torch.arange(max_seq_len).unsqueeze(1)
        div_terms = torch.exp(torch.arange(0, d_model, 2).float() * -math.log(1000) / d_model)
        pe[:, 0::2] = torch.sin(positions * div_terms)
        pe[:, 1::2] = torch.cos(positions * div_terms)

        # Saves 'pe' to the module's state, automatically moving it to the GPU with the model
        self.register_buffer('pe', pe)

    def forward(self, seq_len):
        return self.pe[:seq_len, :]

In [ ]:
class FeedForward(torch.nn.Module):
  def __init__(self,d_model):
    super().__init__()
    self.d_model = d_model
    self.linear1 = torch.nn.Linear(d_model,d_model*4)
    self.linear2 = torch.nn.Linear(d_model*4,d_model)

  def relu(self,x):
    return torch.max(torch.zeros(x.shape).to(x.device),x).to(x.device)

  def forward(self,x):

    x = self.linear1(x)
    x = self.relu(x)
    x = self.linear2(x)

    return x

In [ ]:
class Transformer(torch.nn.Module):
  def __init__(self, d_model, num_heads, dropout = 0.1):
    super().__init__()
    self.attention = Attention(d_model, num_heads)
    self.feed_forward = FeedForward(d_model)
    self.layernorm1 = LayerNorm(d_model)
    self.layernorm2 = LayerNorm(d_model)
    self.dropout = torch.nn.Dropout(dropout)

  def forward(self,x,mask, use_cache = False, kv_cache = None):

    attention_output, new_kv_cache = self.attention(x,x,x,mask, use_cache, kv_cache)

    x = x + self.dropout(attention_output)
    x = self.layernorm1(x)
    ff_out = self.feed_forward(x)
    x = x + self.dropout(ff_out)
    x = self.layernorm2(x)

    return x, new_kv_cache




In [ ]:
device = torch.accelerator.current_accelerator()
device

device(type='cuda')

In [ ]:
class LanguageModel(torch.nn.Module):

  def __init__(self, d_model, num_heads,max_seq_len,vocab_size):
    super().__init__()
    self.embeddings = torch.nn.Embedding(vocab_size,d_model)
    self.positional_encodings = PositionalEncodings(d_model,max_seq_len)
    self.transformer = torch.nn.ModuleList([Transformer(d_model,num_heads) for _ in range(10)])
    self.linear = torch.nn.Linear(d_model,vocab_size)

  def forward(self, x, use_cache = False, past_key_values = None):


    seq_len = x.shape[1]
    past_seq_len = past_key_values[0][0].shape[-2] if past_key_values is not None else 0

    x = self.embeddings(x).to(device)

    total_len = past_seq_len + seq_len
    pe = self.positional_encodings(total_len).to(x.device)

    # 2. Slice out just the positional encodings for the current tokens
    x = x + pe[past_seq_len : past_seq_len + seq_len, :]

    mask = torch.tril(torch.ones(seq_len, seq_len)).to(x.device)
    new_key_values = []
    for i,transformer in enumerate(self.transformer):
      kv = past_key_values[i] if past_key_values is not None else None
      x, new_kv = transformer(x,mask, use_cache, kv)
      if use_cache:
        new_key_values.append(new_kv)

    x = self.linear(x)
    if use_cache:
      return x, new_key_values
    return x




In [ ]:
batch_size = 2
seq_len = 10
max_seq_len = 256
vocab_size = 65
d_model = 16
num_heads = 4


x_dummy = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
model = LanguageModel(d_model, num_heads, max_seq_len, vocab_size).to(device)
logits = model(x_dummy)

print(f"Input shape: {x_dummy.shape} (batch_size, seq_len)")
print(f"Output shape: {logits.shape} (batch_size, seq_len, vocab_size)")
print("LanguageModel forward pass successful!")

Input shape: torch.Size([2, 10]) (batch_size, seq_len)
Output shape: torch.Size([2, 10, 65]) (batch_size, seq_len, vocab_size)
LanguageModel forward pass successful!


In [ ]:
def make_pairs(x):
  ids = x['ids']
  return {
      'input_ids': ids[:-1],
      'labels': ids[1:]
  }

def tokenization(d, stoi=None):
  d = d.map(lambda x : {'chars' : list(x['Text'])})

  if stoi is None:
    all_chars = set()
    for samples in d:
      all_chars.update(samples['chars'])

    vocabulary = sorted(all_chars)
    vocab_size = len(vocabulary)
    print(f"Vocabulary size: {vocab_size}")
    stoi = {ch : i for i, ch in enumerate(vocabulary)}
  else:
    vocab_size = len(stoi)

  # Use stoi.get(ch, 0) to default to index 0 if a character is unseen in the train set
  d = d.map(lambda x : {'ids' : [stoi.get(ch, 0) for ch in x['chars']]})
  d = d.map(make_pairs)
  seq_len = 100
  all_ids = []
  chunks = []
  for x in d:
    all_ids.extend(x['ids'])

  for i in tqdm.tqdm(range(0, len(all_ids) - seq_len)):
    x = all_ids[i : i + seq_len]
    y = all_ids[i+1 : i + seq_len + 1]

    chunks.append({
      'input_ids': torch.tensor(x, dtype=torch.long),
      'labels': torch.tensor(y, dtype=torch.long)
    })
  return chunks, d, vocab_size, stoi


In [ ]:

def byte_level_tokenize(text) -> list:
  byte_tokens = list(text.encode('utf-8'))
  return byte_tokens
def byte_level_detokenize(byte_tokens) -> str:
  text = bytes(byte_tokens).decode('utf-8')
  return "".join(text)

def get_byte_vocab(tokens):
  vocab = Counter()

  for pair in zip(tokens[:-1], tokens[1:]):
    vocab[pair] += 1
  return vocab

def get_most_frequent_pair(vocab):

  common = vocab.most_common()

  if len(common)>0:
    return common[0][0]
  return None

def merge_pairs(tokens, pair, next_id):

  idx = 0
  a,b =pair
  updated_tokens = []
  while idx < len(tokens):

    if idx< len(tokens) - 1 and tokens[idx] == a and tokens[idx + 1]==b:
      updated_tokens.append(next_id)
      idx+=2
    else:
      updated_tokens.append(tokens[idx])
      idx+=1
  return updated_tokens



In [ ]:
class BPE():

  def __init__(self, vocab_size, dest_folder):
    self.vocab_size = vocab_size
    self.pad_id = 256
    self.unk_id = 257
    self.bos_id = 258
    self.eos_id = 259

    self.special_tokens = {
        "<PAD>" : self.pad_id,
        "<UNK>" : self.unk_id,
        "<BOS>" : self.bos_id,
        "<EOS>" : self.eos_id
    }
    self.tokens = []
    self.merges = []
    self.token_to_char = {i : bytes([i]) for i in range(256)}
    self.corpus = ""
    self.dest_folder = dest_folder

  def __call__(self, corpus):
    self.corpus = corpus
    self.tokens,self.merges = self.byte_pair()

  def byte_pair(self):
    tokens = byte_level_tokenize(self.corpus)

    next_id = 260
    merges = []

    for idx in range(self.vocab_size - 260):
      vocab = get_byte_vocab(tokens)
      pair = get_most_frequent_pair(vocab)
      if pair is None:
        break
      self.token_to_char[next_id] = self.token_to_char[pair[0]] + self.token_to_char[pair[1]]
      merges.append((pair, next_id))
      tokens = merge_pairs(tokens,pair,next_id)
      next_id +=  1
    return tokens, merges

  def encode(self, corpus, add_special_tokens = True):
    tokens = byte_level_tokenize(corpus)
    for merge in self.merges:
      tokens = merge_pairs(tokens, merge[0], merge[1])
    if add_special_tokens:
      tokens = [self.bos_id] + tokens + [self.eos_id]
    return tokens

  def decode(self, tokens):
      res = []
      for token in tokens:
          if token == self.pad_id:
            continue
          if token in self.token_to_char:
              byte_val = self.token_to_char[token]
              # Handle cases where the byte sequence isn't valid UTF-8 yet
              try:
                  res.append(byte_val.decode("utf-8", errors="replace"))
              except:
                  res.append("")
          else:
              res.append("<UNK>")
      return "".join(res)


In [ ]:
d = datasets.load_dataset("Trelis/tiny-shakespeare")['train']

bpe = BPE(300, './model')
corpus = "".join(d['Text'])
bpe(corpus)

README.md:   0%|          | 0.00/497 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/119k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/472 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/49 [00:00<?, ? examples/s]

In [ ]:
def prepare_bpe_chunks(dataset, tokenizer,seq_len = 100):
  all_ids =[]
  for sample in tqdm.tqdm(dataset, desc ="Encoding Text"):
    ids = tokenizer.encode(sample['Text'])
    all_ids.extend(ids)
  chunks = []
  for i in range(0, len(all_ids) - seq_len, seq_len//10):
    x = all_ids[i : i + seq_len]
    y = all_ids[i+1 : i+ seq_len + 1]
    chunks.append({
        "input_ids" : torch.tensor(x, dtype = torch.long),
        "labels" : torch.tensor(y, dtype = torch.long)
    })
  return chunks

chunks = prepare_bpe_chunks(d, bpe)

Encoding Text: 100%|██████████| 472/472 [00:02<00:00, 177.38it/s]


In [ ]:
class CharDataset(Dataset):

  def __init__(self, data):

    self.data = data

  def __getitem__(self, index):

    chunk = self.data[index]

    return chunk['input_ids'],chunk['labels']

  def __len__(self):
    return len(self.data)


In [ ]:
class CharDataLoader:

  def __init__(self, dataset, batch_size, shuffle = True):
    self.dataset = dataset
    self.batch_size = batch_size
    self.shuffle = shuffle
  def __len__(self):
    return math.ceil(len(self.dataset)/self.batch_size)
  def __iter__(self):

    indices = torch.arange(len(self.dataset))

    if self.shuffle:
      indices = torch.randperm(len(self.dataset))

    for i in range(0, len(self.dataset),self.batch_size):
      batch_indices = indices[i:i+self.batch_size]
      batch = [self.dataset[idx.item()] for idx in batch_indices]
      batch= self.collate_fn(batch)
      yield batch

  def collate_fn(self, batch):
    x_list,y_list = zip(*batch)
    max_len = max([len(seq) for seq in x_list])

    padded_x = []
    padded_y = []

    for x,y in zip(x_list,y_list):
      pad_amount = max_len - len(x)
      x_padded = torch.nn.functional.pad(x, (0, pad_amount), value = 256)
      y_padded = torch.nn.functional.pad(y, (0, pad_amount), value = 256)
      padded_x.append(x_padded)
      padded_y.append(y_padded)
    return torch.stack(padded_x), torch.stack(padded_y)

In [ ]:
train_dataset = CharDataset(chunks)
train_dataloader = CharDataLoader(train_dataset,32,shuffle = True)


In [ ]:
def softmax(x,dim = -1):
  return torch.nn.functional.softmax(x,dim = dim)


In [ ]:
def train(dataloader, model, loss_fn, optimizer,grad_accumulation_steps , num_epochs):
  size = len(dataloader.dataset)
  scaler = torch.amp.GradScaler('cuda')

  for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for i,batch in enumerate(tqdm.tqdm(dataloader)):

      x, y = batch
      x = x.to(device)
      y = y.to(device)
      with torch.amp.autocast('cuda'):
        logits = model(x)
        loss = loss_fn(logits.transpose(1,2),y)

      scaled_loss = loss/grad_accumulation_steps
      scaler.scale(scaled_loss).backward()

      total_loss += loss.item()

      if (i + 1) % grad_accumulation_steps == 0 or (i + 1) == len(dataloader):
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader)}")





In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [ ]:
model = LanguageModel(256, 8, 100, bpe.vocab_size).to(device)
model = torch.compile(model)
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=256)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

train(train_dataloader, model, loss_fn, optimizer, 4, 10)

100%|██████████| 2742/2742 [00:18<00:00, 145.83it/s]


Epoch 1/10, Loss: 4.247398812956918


100%|██████████| 2742/2742 [00:18<00:00, 146.82it/s]


Epoch 2/10, Loss: 4.240230533521946


100%|██████████| 2742/2742 [00:18<00:00, 145.00it/s]


...

Epoch 8/10, Loss: 4.236973238200014


100%|██████████| 2742/2742 [00:18<00:00, 148.55it/s]


Epoch 9/10, Loss: 4.2369442033037314


100%|██████████| 2742/2742 [00:18<00:00, 148.31it/s]

Epoch 10/10, Loss: 4.23687406198375


In [ ]:
save_path = '/content/drive/MyDrive/language_model_weights.pth'
torch.save(model, '/content/drive/MyDrive/entire_model.pth')
torch.save(model.state_dict(),save_path)
print(f"Model saved to  {save_path}")

Model saved to  /content/drive/MyDrive/language_model_weights.pth


In [ ]:
tokenizer_path = '/content/drive/MyDrive/BPE_tokenizer.pth'
torch.save(bpe, tokenizer_path)
print(f"BPE tokenizer saved to {tokenizer_path}")

BPE tokenizer saved to /content/drive/MyDrive/BPE_tokenizer.pth


In [ ]:
test = datasets.load_dataset("Trelis/tiny-shakespeare")['test']
test_chunks = prepare_bpe_chunks(test, bpe)


Encoding Text: 100%|██████████| 49/49 [00:00<00:00, 190.53it/s]


In [ ]:
test_dataset = CharDataset(test_chunks)
test_dataloader = CharDataLoader(test_dataset, 32, shuffle=True)


In [ ]:
def evaluate(model, dataloader):
  model.eval()
  accuracy = 0
  total_loss = 0
  predictions = []
  for batch in dataloader:
    x, y = batch
    x = x.to(device)
    y = y.to(device)
    with torch.no_grad():
      logits = model(x)
    probs = torch.argmax(logits,dim = -1)
    outputs = softmax(probs)
    predictions.append(outputs)
    accuracy+= torch.sum(outputs == y).item()
  return accuracy/len(dataloader.dataset), predictions

In [ ]:
def evaluate(model, dataloader):
  model.eval()
  accuracy = 0
  total_loss = 0
  predictions = []
  total_elements = 0
  loss_fn = torch.nn.CrossEntropyLoss()

  for batch in dataloader:
    x, y = batch
    x = x.to(device)
    y = y.to(device)
    with torch.no_grad():
      logits = model(x)
      # Compute loss for perplexity calculation
      loss = loss_fn(logits.transpose(1, 2), y)
      total_loss += loss.item() * x.size(0)

    outputs = torch.argmax(logits, dim=-1)
    predictions.append(outputs)
    accuracy += torch.sum(outputs == y).item()
    total_elements += y.numel()

  avg_loss = total_loss / len(dataloader.dataset)
  perplexity = math.exp(avg_loss)
  acc = accuracy / total_elements

  return acc, perplexity, predictions

acc, perplexity, predictions = evaluate(model, test_dataloader)
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test Perplexity: {perplexity:.4f}")


Test Accuracy: 4.43%
Test Perplexity: 70.5552


In [ ]:
test[0]

{'Text': "TRANIO:\nIs this your speeding? nay, then, good night our part!\n\nPETRUCHIO:\nBe patient, gentlemen; I choose her for myself:\nIf she and I be pleased, what's that to you?\n'Tis bargain'd 'twixt us twain, being alone,\nThat she shall still be curst in company.\nI tell you, 'tis incredible to believe\nHow much she loves me: O, the kindest Kate!\nShe hung about my neck; and kiss on kiss\nShe vied so fast, protesting oath on oath,\nThat in a twink she won me to her love.\nO, you are novices! 'tis a world to see,\nHow tame, when men and women are alone,\nA meacock wretch can make the curstest shrew.\nGive me thy hand, Kate: I will unto Venice,\nTo buy apparel 'gainst the wedding-day.\nProvide the feast, father, and bid the guests;\nI will be sure my Katharina shall be fine.\n\nBAPTISTA:\nI know not what to say: but give me your hands;\nGod send you joy, Petruchio! 'tis a match.\n\nGREMIO:\nAmen, say we: we will be witnesses.\n\nPETRUCHIO:\nFather, and wife, and gentlemen, adieu;

In [ ]:
sample = datasets.Dataset.from_dict({"Text": ["Is this your speeding? nay, then, good night our part!\n\nPETRUCHIO:\nBe patient, gentlemen; I choose her for myself:\nIf she and I be pleased, what's that to you?\n'Tis bargain'd 'twixt us twain, being alone,\nThat she shall still be curst in company"]})




chunks_p = prepare_bpe_chunks(sample, bpe)


# print(f"Generated {chunks_p} chunks.")

Encoding Text: 100%|██████████| 1/1 [00:00<00:00, 2035.08it/s]


In [ ]:
testing_dataset = CharDataset(chunks_p)
testing_dataloader = CharDataLoader(testing_dataset, 1, shuffle=False)

In [ ]:
acc,ppl, prediction = evaluate(model,testing_dataloader)
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test Perplexity: {ppl:.4f}")

Test Accuracy: 4.88%
Test Perplexity: 69.1646


In [ ]:
prediction[0]

tensor([[32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32,
         32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32,
         32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32,
         32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32,
         32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32,
         32, 32, 32, 32, 32, 32, 32, 32, 32, 32]], device='cuda:0')

In [ ]:
test[0]

{'Text': "TRANIO:\nIs this your speeding? nay, then, good night our part!\n\nPETRUCHIO:\nBe patient, gentlemen; I choose her for myself:\nIf she and I be pleased, what's that to you?\n'Tis bargain'd 'twixt us twain, being alone,\nThat she shall still be curst in company.\nI tell you, 'tis incredible to believe\nHow much she loves me: O, the kindest Kate!\nShe hung about my neck; and kiss on kiss\nShe vied so fast, protesting oath on oath,\nThat in a twink she won me to her love.\nO, you are novices! 'tis a world to see,\nHow tame, when men and women are alone,\nA meacock wretch can make the curstest shrew.\nGive me thy hand, Kate: I will unto Venice,\nTo buy apparel 'gainst the wedding-day.\nProvide the feast, father, and bid the guests;\nI will be sure my Katharina shall be fine.\n\nBAPTISTA:\nI know not what to say: but give me your hands;\nGod send you joy, Petruchio! 'tis a match.\n\nGREMIO:\nAmen, say we: we will be witnesses.\n\nPETRUCHIO:\nFather, and wife, and gentlemen, adieu;

In [ ]:
bpe.decode(prediction[0][0].tolist())

'                                                                                                    '

In [ ]:
from numpy._core.defchararray import index

def generate_text(model, tokenizer, prompt, max_new_tokens=250, temperature=0.4, top_k=200, top_p=0.9):
    model.eval()

    # Store the full generated sequence in a list
    full_sequence_ids = tokenizer.encode(prompt, add_special_tokens=True)
    if full_sequence_ids[-1] == tokenizer.eos_id:
        full_sequence_ids = full_sequence_ids[:-1]

    x = torch.tensor(full_sequence_ids, dtype=torch.long).unsqueeze(0).to(device)

    past_key_values = None

    for _ in range(max_new_tokens):
        if past_key_values is not None and past_key_values[0][0].shape[-2] >= model.positional_encodings.pe.shape[0]:
            print("Reached model's maximum context length. Stopping generation.")
            break
        with torch.no_grad():
            # Pass use_cache and past_key_values to the model
            outputs = model(x, use_cache=True, past_key_values=past_key_values)
            logits, past_key_values = outputs

            # Get the logits for the very last token in the sequence
            last_token_logits = logits[:, -1, :] / temperature
            probs = torch.nn.functional.softmax(last_token_logits, dim=-1)

            if top_k > 0:
                top_k_probs, top_k_indices = torch.topk(probs, k=top_k, dim=-1)
                mask = torch.zeros_like(probs, dtype=torch.bool)
                mask.scatter_(dim=-1, index=top_k_indices, value=True)
                probs[~mask] = 0.0
            if top_p < 1.0:
                sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                cumulative_sum = torch.cumsum(sorted_probs, dim=-1)
                cutoff_indices = cumulative_sum > top_p
                if torch.any(cutoff_indices):
                    cutoff_index = torch.argmax(cutoff_indices.to(torch.int)).item()
                    if cutoff_index == 0:
                        cutoff_index = 1
                    bad_indices = sorted_indices[:, cutoff_index:]
                    probs[:, bad_indices] = 0.0

            prob_sum = torch.sum(probs)
            if prob_sum > 0.0:
                probs = probs / prob_sum
            else:
                probs = torch.ones_like(probs) / probs.shape[-1]

            next_token_id = torch.multinomial(probs, num_samples=1)

            # --- THE MAGIC HAPPENS HERE ---
            # Instead of appending to 'x' and sending a huge tensor back in,
            # 'x' becomes ONLY the newly generated token.
            x = next_token_id

            # Keep track of the full sequence for decoding later
            full_sequence_ids.append(next_token_id.item())

    return tokenizer.decode(full_sequence_ids)

In [ ]:
generate_text(model, bpe, "Is this your speeding? what")

Reached model's maximum context length. Stopping generation.


'<UNK>Is this your speeding? whatiee t eidt  taIe rle leu a la e  ed eitasthuIe enieor eit eais ce siatl \naeid meesemi htlie t'

In [ ]:
import gradio as gr

def gradio_generate(prompt):
  return generate_text(model, bpe, prompt)
demo = gr.Interface(gradio_generate,inputs="textbox",outputs="textbox")

demo.launch(share = True,  debug = True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://285711811716ace251.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://285711811716ace251.gradio.live
